# Task 1 — Idiom Detection Demo

This notebook loads the fine-tuned RoBERTa model from the saved Task 1 artifacts and predicts whether a sentence is used:

- **Idiomatic**
- **Literal**

The notebook is designed to work from the project repository without retraining.

In [11]:
# 1.1 Install missing libraries and define paths

import sys
import importlib
import subprocess
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    """
    Import a package if available.
    If missing, install it silently, then import it.
    """
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# Required libraries for the demo notebook
ensure_package("torch")
ensure_package("transformers")

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Detect compute device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Resolve project root relative to notebook location
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Local saved model path inside the repository
MODEL_DIR = PROJECT_ROOT / "artifacts" / "task1" / "models" / "roberta"

print("Using device:", DEVICE)
print("Project root:", PROJECT_ROOT.resolve())
print("Model path  :", MODEL_DIR.resolve())
print("Model exists:", MODEL_DIR.exists())

Using device: cuda
Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
Model path  : C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\artifacts\task1\models\roberta
Model exists: True


## 2. Load Tokenizer and Model

Load the saved RoBERTa tokenizer and fine-tuned classification model from the local artifacts directory.

In [12]:
# 2.1 Load tokenizer and model

# Load tokenizer that matches the saved model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# Load fine-tuned sequence classification model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

# Move model to the selected device and switch to inference mode
model = model.to(DEVICE)
model.eval()

print("Model loaded successfully.")
print("Model type:", model.__class__.__name__)

Model loaded successfully.
Model type: RobertaForSequenceClassification


## 3. Prediction Function

Define a reusable function that takes a sentence and predicts whether it is used idiomatically or literally, along with a confidence score.

In [13]:
# 3.1 Prediction function

import torch

LABEL_MAP = {
    0: "Literal",
    1: "Idiomatic"
}

def predict_idiom_usage(text):
    """
    Predict whether a sentence is idiomatic or literal.

    Returns:
        dict with prediction and confidence
    """
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move to device
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item())

    return {
        "text": text,
        "prediction": LABEL_MAP[pred_id],
        "confidence": round(confidence, 4)
    }

## 4. Example Predictions

We test the model on a set of example sentences containing both idiomatic and literal usages.

In [14]:
# 4.1 Example predictions

examples = [
    "He finally kicked the bucket after years of illness.",
    "He kicked the bucket across the yard.",
    "She spilled the beans about the surprise party.",
    "She accidentally spilled the beans on the floor.",
    "This project is just a piece of cake.",
    "She cut a piece of cake for everyone."
]

for text in examples:
    result = predict_idiom_usage(text)

    print("Text       :", result["text"])
    print("Prediction :", result["prediction"])
    print("Confidence :", result["confidence"])
    print("-" * 80)

Text       : He finally kicked the bucket after years of illness.
Prediction : Idiomatic
Confidence : 0.9996
--------------------------------------------------------------------------------
Text       : He kicked the bucket across the yard.
Prediction : Literal
Confidence : 0.8612
--------------------------------------------------------------------------------
Text       : She spilled the beans about the surprise party.
Prediction : Idiomatic
Confidence : 0.9986
--------------------------------------------------------------------------------
Text       : She accidentally spilled the beans on the floor.
Prediction : Literal
Confidence : 0.8617
--------------------------------------------------------------------------------
Text       : This project is just a piece of cake.
Prediction : Idiomatic
Confidence : 0.9996
--------------------------------------------------------------------------------
Text       : She cut a piece of cake for everyone.
Prediction : Idiomatic
Confidence : 0.9365

## 5. Interactive Testing

Enter any sentence and let the model predict whether the usage is idiomatic or literal.
Type `exit` to stop.

In [15]:
# 5.1 Interactive testing loop (improved)

EXIT_COMMANDS = {"exit", "stop", "quit", "q"}

while True:
    text = input("Enter a sentence (type 'exit' / 'stop' / 'quit' to end): ").strip()

    # Exit conditions
    if text.lower() in EXIT_COMMANDS:
        print("Demo ended.")
        break

    # Empty input handling
    if not text:
        print("No input detected. Exiting demo.")
        break

    result = predict_idiom_usage(text)

    print("Prediction :", result["prediction"])
    print("Confidence :", result["confidence"])
    print("-" * 80)

Enter a sentence (type 'exit' / 'stop' / 'quit' to end): spill the tea
Prediction : Idiomatic
Confidence : 0.9987
--------------------------------------------------------------------------------
Enter a sentence (type 'exit' / 'stop' / 'quit' to end): stop
Demo ended.
